# Amharic Speech-to-Speech Assistant — Gemma 4 E4B

Complete ASR → LLM → TTS pipeline on a single Colab T4.

**Models**
- ASR: `snapwre/hohe-asr-amharic`
- LLM: `google/gemma-4-E4B-it` (4-bit NF4)
- TTS: `african-low-resource/omnivoice-amharic`

**Two run modes**
- `time_sliced=True`: Gemma stops after its first stable phrase, OmniVoice speaks that phrase while Gemma is paused, then Gemma continues and the full answer is synthesized. This gives a measurable `first_audio_ready` time without running Gemma and OmniVoice on the GPU at the same time.
- `time_sliced=False`: plain ASR → full LLM answer → TTS. Simpler baseline.

**Sections**
1. Environment setup
2. Configuration
3. Hugging Face authentication (optional)
4. ASR
5. LLM
6. TTS
7. Pipeline
8. Gradio demo
9. Evaluation helper

## 1. Environment Setup

Use a Colab **T4 GPU** runtime. If an older `transformers` was already imported in this kernel, restart the session before re-running from the top.

`bitsandbytes` is required: Gemma 4 E4B is about 16 GB in bf16 (roughly 8B parameters including per-layer embeddings), which does not fit a 15 GB T4 next to ASR and TTS, so it is loaded in 4-bit.

In [1]:
!nvidia-smi
!pip install -q "transformers==5.18.0" accelerate sentencepiece librosa gradio soundfile pydub tensorboardX webdataset jiwer pandas
!pip install -q -U bitsandbytes
!pip install -q --no-deps omnivoice==0.2.1

Fri Oct  2 23:08:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import gc
import re
import os
import time
import uuid
import tempfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import display

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU."

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


def vram_report(tag=""):
    alloc = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM {tag}: {alloc:.2f} GB allocated / {total:.2f} GB total")


def free_cuda():
    gc.collect()
    torch.cuda.empty_cache()


class Timer:
    """Wall-clock timer that synchronizes CUDA on entry and exit."""

    def __enter__(self):
        torch.cuda.synchronize()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        self.elapsed = time.perf_counter() - self._t0
        return False

Torch: 2.11.0+cu130
Transformers: 5.18.0
GPU: Tesla T4
VRAM total: 15.64 GB


## 2. Configuration

All tunable values live here. For a fair comparison with other pipelines, keep ASR, TTS, prompt and decoding settings fixed.

In [3]:
# ---- Models ------------------------------------------------------------------
ASR_MODEL_ID = "snapwre/hohe-asr-amharic"
LLM_MODEL_ID = "google/gemma-4-E4B-it"
TTS_MODEL_ID = "african-low-resource/omnivoice-amharic"

# ---- ASR ---------------------------------------------------------------------
ASR_SAMPLE_RATE = 16000
ASR_DTYPE = torch.float16          # set torch.float32 if transcripts degrade

# ---- LLM decoding ------------------------------------------------------------
LLM_MAX_NEW_TOKENS = 200           # Amharic is token-heavy; lower = faster
LLM_DO_SAMPLE = False              # greedy = reproducible measurements
LLM_TEMPERATURE = 0.7              # used only when LLM_DO_SAMPLE = True
LLM_TOP_P = 0.95
LLM_TOP_K = 64
LLM_REPETITION_PENALTY = 1.05

# ---- Time-sliced first phrase ------------------------------------------------
FIRST_PHRASE_MAX_NEW_TOKENS = 48
FIRST_PHRASE_MIN_WORDS = 2

# ---- TTS ---------------------------------------------------------------------
TTS_STEPS_FIRST = 8                # low step count for the first phrase (speed)
TTS_STEPS_FULL = 16                # starting point for the full answer; raise toward 32 for quality
TTS_GUIDANCE = 2.0
TTS_SEED = 42                      # fixed seed keeps the auto-voice consistent across chunks

FALLBACK_REPLY = "ሰላም! እንዴት ልረዳህ እችላለሁ?"

SYSTEM_PROMPT = (
    "You are a helpful voice assistant that always replies in Amharic (Ge'ez script). "
    "Keep replies natural and conversational, as if speaking out loud - "
    "at most 2-3 short sentences unless the user explicitly asks for more detail. "
    "Give a direct answer rather than repeating the question. "
    "Do not use lists, markdown, emojis or English words. "
    "Do not invent information."
)

OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="s2s_"))
print("Output directory:", OUTPUT_DIR)

Output directory: /tmp/s2s_fohlly2q


## 3. Hugging Face Authentication (optional)

`google/gemma-4-E4B-it` is Apache-2.0 and not gated, so a token is not required. If an `HF_TOKEN` environment variable or Colab secret exists it is used for higher download rate limits.

In [4]:
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Using HF_TOKEN.")
else:
    print("No HF_TOKEN found. Continuing anonymously.")

Using HF_TOKEN.


## 4. Hohe Amharic ASR

In [5]:
from transformers import AutoProcessor, AutoModelForCTC

free_cuda()

asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_ID)
asr_model = AutoModelForCTC.from_pretrained(
    ASR_MODEL_ID,
    dtype=ASR_DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE).eval()

print("ASR loaded:", next(asr_model.parameters()).dtype)
vram_report("after ASR")


def _asr_inputs_to_gpu(inputs):
    return {
        k: v.to(DEVICE, dtype=ASR_DTYPE) if v.is_floating_point() else v.to(DEVICE)
        for k, v in inputs.items()
    }


def clean_asr_text(text):
    return re.sub(r"\[AMH\]", "", text).strip()


def transcribe(audio_path):
    """Returns the cleaned transcript plus preprocessing, inference and total timings."""
    t_start = time.perf_counter()

    audio, _ = librosa.load(audio_path, sr=ASR_SAMPLE_RATE, mono=True)
    inputs = asr_processor(audio, sampling_rate=ASR_SAMPLE_RATE, return_tensors="pt")
    gpu_inputs = _asr_inputs_to_gpu(inputs)
    prep_s = time.perf_counter() - t_start

    with Timer() as t, torch.inference_mode():
        logits = asr_model(**gpu_inputs).logits

    pred_ids = logits.argmax(dim=-1)
    raw = asr_processor.batch_decode(pred_ids)[0]

    return {
        "raw": raw,
        "transcript": clean_asr_text(raw),
        "preprocessing_s": prep_s,
        "inference_s": t.elapsed,
        "total_s": time.perf_counter() - t_start,
    }


# Warm-up so the first real request is not slowed by CUDA initialization.
_dummy = asr_processor(np.zeros(ASR_SAMPLE_RATE, dtype=np.float32),
                       sampling_rate=ASR_SAMPLE_RATE, return_tensors="pt")
with torch.inference_mode():
    _ = asr_model(**_asr_inputs_to_gpu(_dummy)).logits
torch.cuda.synchronize()
print("ASR warm-up complete")

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/32.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/797 [00:00<?, ?it/s]

ASR loaded: torch.float16
VRAM after ASR: 1.21 GB allocated / 15.64 GB total
ASR warm-up complete


## 5. Gemma 4 E4B-it (4-bit NF4)

NF4 quantization with fp16 compute, because the T4 has no native bf16. Thinking mode is disabled to avoid hidden latency. LLM output is sanitized to Ethiopic script so stray foreign-script tokens never reach the TTS model.

In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)

free_cuda()
print(f"Loading {LLM_MODEL_ID} in 4-bit NF4...")
t0 = time.perf_counter()

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, token=HF_TOKEN)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map={"": 0},
    token=HF_TOKEN,
).eval()

print(f"Gemma loaded in {time.perf_counter() - t0:.1f}s")
vram_report("after LLM")

PAD_ID = llm_tokenizer.pad_token_id
if PAD_ID is None:
    PAD_ID = llm_tokenizer.eos_token_id

_eos = llm_model.generation_config.eos_token_id
EOS_IDS = set(_eos if isinstance(_eos, (list, tuple)) else [_eos])
if llm_tokenizer.eos_token_id is not None:
    EOS_IDS.add(llm_tokenizer.eos_token_id)
EOS_IDS.discard(None)

Loading google/gemma-4-E4B-it in 4-bit NF4...


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Gemma loaded in 1047.5s
VRAM after LLM: 10.53 GB allocated / 15.64 GB total


In [7]:
# ---- Text helpers ------------------------------------------------------------
PHRASE_ENDINGS = ("፣", "።", "፤", ".", "!", "?")


def sanitize_to_ethiopic(text):
    """Drops thinking blocks and any non-Ethiopic characters; keeps digits and basic punctuation."""
    text = re.sub(r"<\|channel>.*?<channel\|>", "", text, flags=re.S)
    text = re.sub(r"[^ሀ-፿\s0-9\.\,\?\!]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def get_stable_phrase(text, min_words=2):
    """First phrase ending in punctuation that contains at least min_words words, else None."""
    text = text.strip()
    for i, ch in enumerate(text):
        if ch in PHRASE_ENDINGS:
            candidate = text[: i + 1].strip()
            if len(candidate.split()) >= min_words:
                return candidate
    return None


class StopAtStablePhrase(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length, min_words=2):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
        self.min_words = min_words

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(input_ids[0, self.prompt_length:], skip_special_tokens=True)
        return get_stable_phrase(sanitize_to_ethiopic(text), self.min_words) is not None


# ---- Generation --------------------------------------------------------------
def build_chat_inputs(user_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    try:
        inputs = llm_tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt", enable_thinking=False,
        )
    except Exception:
        # Template rejected the system role: fold it into the user turn.
        inputs = llm_tokenizer.apply_chat_template(
            [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_text}"}],
            add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        )
    return inputs.to(llm_model.device)


def _decode(ids):
    return sanitize_to_ethiopic(llm_tokenizer.decode(ids, skip_special_tokens=True))


def _gen_kwargs(max_new_tokens):
    kw = dict(max_new_tokens=max_new_tokens, repetition_penalty=LLM_REPETITION_PENALTY,
              pad_token_id=PAD_ID, use_cache=True)
    if LLM_DO_SAMPLE:
        kw.update(do_sample=True, temperature=LLM_TEMPERATURE, top_p=LLM_TOP_P, top_k=LLM_TOP_K)
    else:
        kw.update(do_sample=False)
    return kw


def generate_full(user_text):
    """Single-pass generation of the whole answer."""
    inputs = build_chat_inputs(user_text)
    prompt_len = inputs["input_ids"].shape[-1]

    with Timer() as t, torch.inference_mode():
        out = llm_model.generate(**inputs, **_gen_kwargs(LLM_MAX_NEW_TOKENS))

    new_ids = out[0, prompt_len:]
    return {"text": _decode(new_ids), "time": t.elapsed, "new_tokens": int(new_ids.shape[-1])}


def generate_first_phrase(user_text):
    """Generates until the first stable phrase, keeping state so generation can continue."""
    inputs = build_chat_inputs(user_text)
    prompt_len = inputs["input_ids"].shape[-1]
    stopper = StopAtStablePhrase(llm_tokenizer, prompt_len, FIRST_PHRASE_MIN_WORDS)

    kw = _gen_kwargs(FIRST_PHRASE_MAX_NEW_TOKENS)
    with Timer() as t, torch.inference_mode():
        out = llm_model.generate(**inputs, stopping_criteria=StoppingCriteriaList([stopper]), **kw)

    new_ids = out[0, prompt_len:]
    text = _decode(new_ids)
    phrase = get_stable_phrase(text, FIRST_PHRASE_MIN_WORDS) or text

    return {
        "inputs": inputs,
        "output_ids": out,
        "prompt_length": prompt_len,
        "phrase": phrase,
        "text": text,
        "finished": int(new_ids[-1]) in EOS_IDS,
        "time": t.elapsed,
    }


def continue_generation(first_stage):
    """Resumes the same answer after the first phrase and returns the complete text."""
    ids = first_stage["output_ids"]
    original = first_stage["inputs"]
    cont = {"input_ids": ids, "attention_mask": torch.ones_like(ids)}

    if "token_type_ids" in original:
        types = original["token_type_ids"]
        extra = ids.shape[-1] - types.shape[-1]
        cont["token_type_ids"] = torch.cat([types, types[:, -1:].expand(-1, extra)], dim=-1)

    with Timer() as t, torch.inference_mode():
        done = llm_model.generate(**cont, **_gen_kwargs(LLM_MAX_NEW_TOKENS))

    return {"text": _decode(done[0, first_stage["prompt_length"]:]), "time": t.elapsed}


# Warm-up: the first bitsandbytes/CUDA call is slow.
with Timer() as _t:
    _ = generate_full("ሰላም")
print(f"LLM warm-up: {_t.elapsed:.1f}s -> {_['text']}")
vram_report("after LLM warm-up")

LLM warm-up: 3.0s -> ሰላምሽ ይድረስ። እንዴት ልረዳህ እችላለሁ?
VRAM after LLM warm-up: 10.53 GB allocated / 15.64 GB total


## 6. OmniVoice Amharic TTS

In [8]:
from omnivoice import OmniVoice

try:
    from omnivoice import OmniVoiceGenerationConfig
except ImportError:
    from omnivoice.models.omnivoice import OmniVoiceGenerationConfig

free_cuda()
tts_model = OmniVoice.from_pretrained(TTS_MODEL_ID, device_map="cuda:0", dtype=torch.float16)
tts_model.eval()
TTS_SAMPLE_RATE = tts_model.sampling_rate or 24000

print(f"OmniVoice loaded (sampling rate {TTS_SAMPLE_RATE} Hz)")
vram_report("after TTS")


def synthesize(text, num_step, tag):
    """Synthesizes text to a WAV file and returns the path, waveform and timing/RTF."""
    text = text.strip()
    if not text:
        raise ValueError("Empty text passed to TTS.")

    config = OmniVoiceGenerationConfig(num_step=num_step, guidance_scale=TTS_GUIDANCE)
    torch.manual_seed(TTS_SEED)

    with Timer() as t, torch.inference_mode():
        audio = tts_model.generate(text=text, language="Amharic", generation_config=config)

    wave = audio[0]
    if isinstance(wave, torch.Tensor):
        wave = wave.detach().cpu().float().numpy()
    wave = np.asarray(wave).squeeze()

    path = OUTPUT_DIR / f"{tag}_{uuid.uuid4().hex[:8]}.wav"
    sf.write(str(path), wave, TTS_SAMPLE_RATE)

    duration = len(wave) / TTS_SAMPLE_RATE
    return {
        "path": str(path),
        "waveform": wave,
        "time": t.elapsed,
        "duration": duration,
        "rtf": t.elapsed / duration if duration > 0 else float("nan"),
    }


_ = synthesize("ሰላም", TTS_STEPS_FIRST, "warmup")
print("TTS warm-up complete")
vram_report("all models loaded")

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

OmniVoice loaded (sampling rate 24000 Hz)
VRAM after TTS: 12.56 GB allocated / 15.64 GB total
TTS warm-up complete
VRAM all models loaded: 12.56 GB allocated / 15.64 GB total


## 7. End-to-End Pipeline

`run_s2s` returns one dict used by the Gradio demo and the evaluation helper. If the LLM runs out of memory (or returns nothing usable after sanitizing), the fixed fallback reply is spoken and `fallback=True` is set in the result, so it is never silent.

In [9]:
def run_s2s(audio_path, time_sliced=True):
    torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()

    asr = transcribe(audio_path)
    transcript = asr["transcript"]
    if not transcript:
        raise ValueError("No speech was transcribed.")

    first_phrase = None
    first_gen_s = first_tts_s = first_dur = 0.0
    first_audio_path = None
    first_audio_ready_s = None
    continuation_s = 0.0
    fallback = False

    try:
        if time_sliced:
            first = generate_first_phrase(transcript)
            first_phrase, first_gen_s = first["phrase"], first["time"]

            if first_phrase:
                first_audio = synthesize(first_phrase, TTS_STEPS_FIRST, "first")
                first_tts_s, first_dur = first_audio["time"], first_audio["duration"]
                first_audio_path = first_audio["path"]
                first_audio_ready_s = time.perf_counter() - t_start

            if first["finished"]:
                full_text = first["text"]
            else:
                cont = continue_generation(first)
                full_text, continuation_s = cont["text"], cont["time"]
        else:
            gen = generate_full(transcript)
            full_text, continuation_s = gen["text"], gen["time"]
    except torch.cuda.OutOfMemoryError:
        free_cuda()
        fallback, full_text = True, FALLBACK_REPLY
        first_phrase = first_audio_path = first_audio_ready_s = None

    if not full_text:
        fallback, full_text = True, FALLBACK_REPLY

    full_audio = synthesize(full_text, TTS_STEPS_FULL, "full")
    total_s = time.perf_counter() - t_start

    if first_audio_ready_s is None:
        # Without a separate first chunk, the first audible output is the complete answer.
        first_audio_ready_s = total_s

    return {
        "transcript": transcript,
        "first_phrase": first_phrase,
        "full_response": full_text,
        "audio_path": full_audio["path"],
        "first_audio_path": first_audio_path,
        "fallback": fallback,
        "time_sliced": time_sliced,
        "asr_s": asr["total_s"],
        "llm_first_phrase_s": first_gen_s,
        "first_phrase_tts_s": first_tts_s,
        "first_audio_ready_s": first_audio_ready_s,
        "first_audio_duration_s": first_dur,
        "llm_continuation_s": continuation_s,
        "full_tts_s": full_audio["time"],
        "full_tts_rtf": full_audio["rtf"],
        "full_audio_duration_s": full_audio["duration"],
        "total_compute_s": total_s,
        "peak_vram_gb": torch.cuda.max_memory_allocated() / 1e9,
    }


def format_metrics(r):
    lines = [
        f"Mode: {'time-sliced' if r['time_sliced'] else 'single-pass'}",
        f"ASR: {r['asr_s']:.2f} s",
    ]
    if r["time_sliced"]:
        lines += [
            f"LLM first phrase: {r['llm_first_phrase_s']:.2f} s",
            f"TTS first phrase: {r['first_phrase_tts_s']:.2f} s",
        ]
    lines += [
        f"FIRST AUDIO READY: {r['first_audio_ready_s']:.2f} s",
        f"LLM {'continuation' if r['time_sliced'] else 'generation'}: {r['llm_continuation_s']:.2f} s",
        f"TTS full response: {r['full_tts_s']:.2f} s (RTF {r['full_tts_rtf']:.2f})",
        f"Full audio duration: {r['full_audio_duration_s']:.2f} s",
        f"Total compute: {r['total_compute_s']:.2f} s",
        f"Peak VRAM: {r['peak_vram_gb']:.2f} GB",
    ]
    if r["fallback"]:
        lines.append("WARNING: fallback reply used (LLM OOM or empty output).")
    return "\n".join(lines)

## 8. Gradio Voice Demo

Record from the microphone or upload a file. The output audio is the complete synthesized answer; the latency box shows the first-audio measurement separately.

In [10]:
import gradio as gr


def voice_assistant(audio_path, time_sliced):
    if audio_path is None:
        return "", "", "", None, "No audio received."
    try:
        r = run_s2s(audio_path, time_sliced=time_sliced)
        return r["transcript"], r["first_phrase"] or "", r["full_response"], r["audio_path"], format_metrics(r)
    except Exception as e:
        return "", "", "", None, f"ERROR:\n{type(e).__name__}: {e}"


with gr.Blocks() as demo:
    gr.Markdown("# Amharic Speech-to-Speech Assistant (Gemma 4 E4B)")

    input_audio = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Speak in Amharic")
    sliced_box = gr.Checkbox(value=True, label="Time-sliced first phrase")
    run_button = gr.Button("Run")

    transcript_box = gr.Textbox(label="ASR Transcript")
    phrase_box = gr.Textbox(label="First Stable Phrase")
    response_box = gr.Textbox(label="Full LLM Response")
    output_audio = gr.Audio(label="Spoken Response", autoplay=True)
    metrics_box = gr.Textbox(label="Latency", lines=10)

    run_button.click(
        voice_assistant,
        inputs=[input_audio, sliced_box],
        outputs=[transcript_box, phrase_box, response_box, output_audio, metrics_box],
    )

demo.queue().launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3feaa5869ac87d8240.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Evaluation Helper

Runs a fixed set of recorded Amharic clips through the pipeline and returns a DataFrame. If reference transcripts are supplied, ASR WER and CER are computed on text normalized with the Ge'ez homophone folding below. Add human quality ratings for the LLM response and audio separately.

In [11]:
import jiwer

# Fold Ge'ez homophone series onto one representative series (same position in each series).
_FOLD = {}
for _src, _dst in [
    (0x1210, 0x1200),  # ሐ -> ሀ
    (0x1280, 0x1200),  # ኀ -> ሀ
    (0x1220, 0x1230),  # ሠ -> ሰ
    (0x12D0, 0x12A0),  # ዐ -> አ
    (0x1340, 0x1338),  # ፀ -> ጸ
]:
    for _i in range(7):
        _FOLD[_src + _i] = _dst + _i

_PUNCT = {c: " " for c in list(range(0x1360, 0x1369)) + [ord(c) for c in "?!.,-\"'"]}


def normalize_amharic_text(text):
    """Strips the ASR tag, folds homophones and removes punctuation (for WER/CER only)."""
    text = re.sub(r"\[AMH\]", "", text or "")
    text = text.translate(_FOLD).translate(_PUNCT)
    return re.sub(r"\s+", " ", text).strip()


CANDIDATE_NAME = LLM_MODEL_ID


def evaluate_candidate(audio_files, references=None, time_sliced=True, out_csv=None):
    """audio_files: list of paths. references: optional list of ground-truth transcripts, same order."""
    rows = []
    for idx, path in enumerate(audio_files):
        r = run_s2s(path, time_sliced=time_sliced)
        row = {
            "candidate": CANDIDATE_NAME,
            "audio_file": path,
            "transcript": r["transcript"],
            "response": r["full_response"],
            "fallback": r["fallback"],
            "asr_s": r["asr_s"],
            "llm_first_phrase_s": r["llm_first_phrase_s"],
            "first_phrase_tts_s": r["first_phrase_tts_s"],
            "first_audio_ready_s": r["first_audio_ready_s"],
            "llm_continuation_s": r["llm_continuation_s"],
            "full_tts_s": r["full_tts_s"],
            "full_tts_rtf": r["full_tts_rtf"],
            "total_compute_s": r["total_compute_s"],
            "peak_vram_gb": r["peak_vram_gb"],
        }
        if references is not None:
            ref = normalize_amharic_text(references[idx])
            hyp = normalize_amharic_text(r["transcript"])
            row["asr_wer"] = jiwer.wer(ref, hyp)
            row["asr_cer"] = jiwer.cer(ref, hyp)
        rows.append(row)

    df = pd.DataFrame(rows)
    display(df)

    if len(df):
        print("\nCandidate:", CANDIDATE_NAME)
        print("Median first audio ready:", round(df["first_audio_ready_s"].median(), 2), "s")
        print("Median total compute:", round(df["total_compute_s"].median(), 2), "s")
        print("Max peak VRAM:", round(df["peak_vram_gb"].max(), 2), "GB")
        if "asr_wer" in df:
            print("Mean ASR WER:", round(df["asr_wer"].mean(), 3), "| CER:", round(df["asr_cer"].mean(), 3))
        if df["fallback"].any():
            print("WARNING: fallback reply used in", int(df["fallback"].sum()), "run(s).")

    if out_csv:
        df.to_csv(out_csv, index=False)
        print("Saved:", out_csv)
    return df

